
# Roxy notebook example: Compositional bias and amino acid usage inequality descriptors

This notebook is a **reference implementation example** for the **compositional bias and amino acid usage inequality descriptor family** in Roxy.

These descriptors try to quantify whether a sequence shows:

- balanced amino acid usage
- strong enrichment of a few residues
- bias toward residue groups
- deviations from uniformity
- strong asymmetry between related residue classes

They are useful because two sequences can have similar entropy or similar complexity, yet differ substantially in **which residues dominate** and **how unevenly composition is distributed**.

## Covered outputs

This notebook implements examples such as:

- amino acid usage variance
- amino acid usage standard deviation
- coefficient of variation of usage
- maximum residue fraction
- top-2 residue burden
- top-3 residue burden
- dominance gap between most common residues
- compositional skew across residue groups
- Gini-like inequality
- deviation from uniform composition
- KL-divergence against a uniform background
- group asymmetry descriptors
- class-style implementation for later migration into Roxy

The notebook is written as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

from collections import Counter

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "bias_1",
            "bias_2",
            "bias_3",
            "bias_4",
            "bias_5",
            "bias_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,bias_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,bias_2,GGGGGGGGGGGGGGG,B
2,bias_3,KRRKRRKRRKRRDDDDEE,A
3,bias_4,ACDEFGHIKLMNPQRSTVWY,B
4,bias_5,PPPPGSSSSSTTTTNNQQQ,A
5,bias_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA_SET])


def amino_acid_frequencies(seq: str) -> np.ndarray:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return np.array([], dtype=float)
    counts = Counter(seq)
    return np.array([counts.get(aa, 0) / len(seq) for aa in STANDARD_AA], dtype=float)


def group_fraction(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def safe_ratio(a: float, b: float) -> float:
    if b == 0:
        return np.nan
    return a / b


def safe_log2(x: float) -> float:
    if x <= 0:
        return 0.0
    return float(np.log2(x))


def gini_like_inequality(freqs: np.ndarray) -> float:
    if len(freqs) == 0:
        return np.nan
    sorted_freqs = np.sort(freqs)
    n = len(sorted_freqs)
    if sorted_freqs.sum() == 0:
        return np.nan
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * sorted_freqs) / (n * np.sum(sorted_freqs))) - (n + 1) / n)


def kl_divergence_uniform(freqs: np.ndarray) -> float:
    if len(freqs) == 0:
        return np.nan
    uniform = 1.0 / len(freqs)
    kl = 0.0
    for p in freqs:
        if p > 0:
            kl += p * np.log2(p / uniform)
    return float(kl)


def l1_deviation_uniform(freqs: np.ndarray) -> float:
    if len(freqs) == 0:
        return np.nan
    uniform = 1.0 / len(freqs)
    return float(np.sum(np.abs(freqs - uniform)))


def dominance_gap(freqs: np.ndarray) -> float:
    if len(freqs) == 0:
        return np.nan
    sorted_freqs = np.sort(freqs)[::-1]
    if len(sorted_freqs) < 2:
        return np.nan
    return float(sorted_freqs[0] - sorted_freqs[1])


def top_k_burden(freqs: np.ndarray, k: int) -> float:
    if len(freqs) == 0:
        return np.nan
    sorted_freqs = np.sort(freqs)[::-1]
    return float(np.sum(sorted_freqs[:k]))


def coefficient_of_variation(freqs: np.ndarray) -> float:
    if len(freqs) == 0:
        return np.nan
    mean = np.mean(freqs)
    std = np.std(freqs, ddof=0)
    if mean == 0:
        return np.nan
    return float(std / mean)


## Core descriptor function

In [5]:

def compositional_bias_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    freqs = amino_acid_frequencies(seq)

    out = {
        "bias_length": len(seq),
        "bias_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    out["bias_usage_mean"] = float(np.mean(freqs))
    out["bias_usage_std"] = float(np.std(freqs, ddof=0))
    out["bias_usage_var"] = float(np.var(freqs, ddof=0))
    out["bias_usage_cv"] = coefficient_of_variation(freqs)
    out["bias_max_residue_fraction"] = float(np.max(freqs))
    out["bias_min_residue_fraction"] = float(np.min(freqs))
    out["bias_top2_burden"] = top_k_burden(freqs, 2)
    out["bias_top3_burden"] = top_k_burden(freqs, 3)
    out["bias_top5_burden"] = top_k_burden(freqs, 5)
    out["bias_dominance_gap"] = dominance_gap(freqs)
    out["bias_gini_like_inequality"] = gini_like_inequality(freqs)
    out["bias_kl_div_uniform"] = kl_divergence_uniform(freqs)
    out["bias_l1_dev_uniform"] = l1_deviation_uniform(freqs)

    # Group-level asymmetry and skew
    positive = group_fraction(seq, AA_GROUPS["positive"])
    negative = group_fraction(seq, AA_GROUPS["negative"])
    polar = group_fraction(seq, AA_GROUPS["polar"])
    nonpolar = group_fraction(seq, AA_GROUPS["nonpolar"])
    aromatic = group_fraction(seq, AA_GROUPS["aromatic"])
    aliphatic = group_fraction(seq, AA_GROUPS["aliphatic"])
    hydrophobic = group_fraction(seq, AA_GROUPS["hydrophobic"])
    hydrophilic = group_fraction(seq, AA_GROUPS["hydrophilic"])
    disorder = group_fraction(seq, AA_GROUPS["disorder_promoting"])
    order = group_fraction(seq, AA_GROUPS["order_promoting"])

    out["bias_positive_negative_skew"] = positive - negative
    out["bias_hydrophobic_hydrophilic_skew"] = hydrophobic - hydrophilic
    out["bias_polar_nonpolar_skew"] = polar - nonpolar
    out["bias_aromatic_aliphatic_skew"] = aromatic - aliphatic
    out["bias_disorder_order_skew"] = disorder - order

    out["bias_positive_negative_ratio"] = safe_ratio(positive, negative)
    out["bias_hydrophobic_hydrophilic_ratio"] = safe_ratio(hydrophobic, hydrophilic)
    out["bias_polar_nonpolar_ratio"] = safe_ratio(polar, nonpolar)
    out["bias_aromatic_aliphatic_ratio"] = safe_ratio(aromatic, aliphatic)
    out["bias_disorder_order_ratio"] = safe_ratio(disorder, order)

    return out


## Functional usage on one sequence

In [6]:

example = compositional_bias_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:18]


[('bias_length', 24),
 ('bias_valid_residue_count', 24),
 ('bias_usage_mean', 0.049999999999999996),
 ('bias_usage_std', 0.05368374469468802),
 ('bias_usage_var', 0.002881944444444444),
 ('bias_usage_cv', 1.0736748938937606),
 ('bias_max_residue_fraction', 0.16666666666666666),
 ('bias_min_residue_fraction', 0.0),
 ('bias_top2_burden', 0.3333333333333333),
 ('bias_top3_burden', 0.4583333333333333),
 ('bias_top5_burden', 0.6666666666666666),
 ('bias_dominance_gap', 0.0),
 ('bias_gini_like_inequality', 0.5541666666666667),
 ('bias_kl_div_uniform', 0.883206219346495),
 ('bias_l1_dev_uniform', 0.8333333333333333),
 ('bias_positive_negative_skew', 0.16666666666666666),
 ('bias_hydrophobic_hydrophilic_skew', 0.20833333333333337),
 ('bias_polar_nonpolar_skew', -0.08333333333333331)]

## Apply compositional bias descriptors to the full dataset

In [7]:

df_bias = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(compositional_bias_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_bias.head()


,sequence_id,sequence,label,bias_length,bias_valid_residue_count,bias_usage_mean,bias_usage_std,bias_usage_var,bias_usage_cv,bias_max_residue_fraction,...,bias_positive_negative_skew,bias_hydrophobic_hydrophilic_skew,bias_polar_nonpolar_skew,bias_aromatic_aliphatic_skew,bias_disorder_order_skew,bias_positive_negative_ratio,bias_hydrophobic_hydrophilic_ratio,bias_polar_nonpolar_ratio,bias_aromatic_aliphatic_ratio,bias_disorder_order_ratio
0,bias_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.05,5.368374e-02,2.881944e-03,1.073675e+00,0.166667,...,0.166667,0.208333,-0.083333,-0.083333,-0.083333,NaN,1.555556,0.846154,0.75,0.833333
1,bias_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.05,2.179449e-01,4.750000e-02,4.358899e+00,1.000000,...,0.000000,0.000000,-1.000000,0.000000,1.000000,NaN,NaN,0.000000,NaN,NaN
2,bias_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.05,1.137194e-01,1.293210e-02,2.274388e+00,0.444444,...,0.333333,-1.000000,1.000000,0.000000,0.777778,2.0,0.000000,NaN,NaN,NaN
3,bias_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.05,6.938894e-18,4.814825e-35,1.387779e-16,0.050000,...,0.050000,0.000000,0.200000,-0.050000,0.000000,1.5,1.000000,1.500000,0.80,1.000000
4,bias_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.05,8.563758e-02,7.333795e-03,1.712752e+00,0.263158,...,0.000000,-0.736842,0.473684,0.000000,0.578947,NaN,0.000000,2.800000,NaN,6.500000


## Inspect bias descriptor columns

In [8]:

bias_cols = [c for c in df_bias.columns if c.startswith("bias_") and c not in {"bias_length", "bias_valid_residue_count"}]
len(bias_cols), bias_cols[:16]


(23,
 ['bias_usage_mean',
  'bias_usage_std',
  'bias_usage_var',
  'bias_usage_cv',
  'bias_max_residue_fraction',
  'bias_min_residue_fraction',
  'bias_top2_burden',
  'bias_top3_burden',
  'bias_top5_burden',
  'bias_dominance_gap',
  'bias_gini_like_inequality',
  'bias_kl_div_uniform',
  'bias_l1_dev_uniform',
  'bias_positive_negative_skew',
  'bias_hydrophobic_hydrophilic_skew',
  'bias_polar_nonpolar_skew'])

In [9]:

df_bias[
    [
        "sequence_id",
        "bias_usage_std",
        "bias_max_residue_fraction",
        "bias_top3_burden",
        "bias_dominance_gap",
        "bias_gini_like_inequality",
        "bias_positive_negative_skew",
        "bias_hydrophobic_hydrophilic_ratio",
    ]
]


,sequence_id,bias_usage_std,bias_max_residue_fraction,bias_top3_burden,bias_dominance_gap,bias_gini_like_inequality,bias_positive_negative_skew,bias_hydrophobic_hydrophilic_ratio
0,bias_1,5.368374e-02,0.166667,0.458333,0.000000,5.541667e-01,0.166667,1.555556
1,bias_2,2.179449e-01,1.000000,1.000000,1.000000,9.500000e-01,0.000000,NaN
2,bias_3,1.137194e-01,0.444444,0.888889,0.222222,8.500000e-01,0.333333,0.000000
3,bias_4,6.938894e-18,0.050000,0.150000,0.000000,-2.220446e-16,0.050000,1.000000
4,bias_5,8.563758e-02,0.263158,0.684211,0.052632,7.710526e-01,0.000000,0.000000
5,bias_6,4.117052e-02,0.142857,0.333333,0.047619,4.452381e-01,0.095238,0.500000


## Dataset-level summary

In [10]:

bias_summary = (
    df_bias[bias_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

bias_summary.head(15)


,descriptor,mean_value
0,bias_disorder_order_ratio,2.440476
1,bias_positive_negative_ratio,1.833333
2,bias_usage_cv,1.707187
3,bias_kl_div_uniform,1.700893
4,bias_polar_nonpolar_ratio,1.295897
5,bias_l1_dev_uniform,1.061905
6,bias_top5_burden,0.731307
7,bias_hydrophobic_hydrophilic_ratio,0.611111
8,bias_gini_like_inequality,0.595076
9,bias_top3_burden,0.585794


## Sanity checks

In [11]:

assert "bias_usage_std" in df_bias.columns
assert "bias_top3_burden" in df_bias.columns
assert "bias_gini_like_inequality" in df_bias.columns
assert "bias_kl_div_uniform" in df_bias.columns
assert "bias_positive_negative_skew" in df_bias.columns
assert df_bias["bias_length"].min() > 0

print(f"Number of compositional bias descriptor columns: {len(bias_cols)}")
print("Compositional bias descriptor checks passed.")


Number of compositional bias descriptor columns: 23
Compositional bias descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class CompositionalBiasDescriptors:
    """Example class-style bias/inequality implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return compositional_bias_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


bias_transformer = CompositionalBiasDescriptors()
bias_matrix = bias_transformer.transform(df_demo["sequence"].tolist())
bias_matrix.head()


,bias_length,bias_valid_residue_count,bias_usage_mean,bias_usage_std,bias_usage_var,bias_usage_cv,bias_max_residue_fraction,bias_min_residue_fraction,bias_top2_burden,bias_top3_burden,...,bias_positive_negative_skew,bias_hydrophobic_hydrophilic_skew,bias_polar_nonpolar_skew,bias_aromatic_aliphatic_skew,bias_disorder_order_skew,bias_positive_negative_ratio,bias_hydrophobic_hydrophilic_ratio,bias_polar_nonpolar_ratio,bias_aromatic_aliphatic_ratio,bias_disorder_order_ratio
0,24,24,0.05,5.368374e-02,2.881944e-03,1.073675e+00,0.166667,0.00,0.333333,0.458333,...,0.166667,0.208333,-0.083333,-0.083333,-0.083333,NaN,1.555556,0.846154,0.75,0.833333
1,15,15,0.05,2.179449e-01,4.750000e-02,4.358899e+00,1.000000,0.00,1.000000,1.000000,...,0.000000,0.000000,-1.000000,0.000000,1.000000,NaN,NaN,0.000000,NaN,NaN
2,18,18,0.05,1.137194e-01,1.293210e-02,2.274388e+00,0.444444,0.00,0.666667,0.888889,...,0.333333,-1.000000,1.000000,0.000000,0.777778,2.0,0.000000,NaN,NaN,NaN
3,20,20,0.05,6.938894e-18,4.814825e-35,1.387779e-16,0.050000,0.05,0.100000,0.150000,...,0.050000,0.000000,0.200000,-0.050000,0.000000,1.5,1.000000,1.500000,0.80,1.000000
4,19,19,0.05,8.563758e-02,7.333795e-03,1.712752e+00,0.263158,0.00,0.473684,0.684211,...,0.000000,-0.736842,0.473684,0.000000,0.578947,NaN,0.000000,2.800000,NaN,6.500000


## Merge transformer output back to the dataset

In [13]:

df_bias_class = pd.concat([df_demo, bias_matrix], axis=1)
df_bias_class.head()


,sequence_id,sequence,label,bias_length,bias_valid_residue_count,bias_usage_mean,bias_usage_std,bias_usage_var,bias_usage_cv,bias_max_residue_fraction,...,bias_positive_negative_skew,bias_hydrophobic_hydrophilic_skew,bias_polar_nonpolar_skew,bias_aromatic_aliphatic_skew,bias_disorder_order_skew,bias_positive_negative_ratio,bias_hydrophobic_hydrophilic_ratio,bias_polar_nonpolar_ratio,bias_aromatic_aliphatic_ratio,bias_disorder_order_ratio
0,bias_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.05,5.368374e-02,2.881944e-03,1.073675e+00,0.166667,...,0.166667,0.208333,-0.083333,-0.083333,-0.083333,NaN,1.555556,0.846154,0.75,0.833333
1,bias_2,GGGGGGGGGGGGGGG,B,15,15,0.05,2.179449e-01,4.750000e-02,4.358899e+00,1.000000,...,0.000000,0.000000,-1.000000,0.000000,1.000000,NaN,NaN,0.000000,NaN,NaN
2,bias_3,KRRKRRKRRKRRDDDDEE,A,18,18,0.05,1.137194e-01,1.293210e-02,2.274388e+00,0.444444,...,0.333333,-1.000000,1.000000,0.000000,0.777778,2.0,0.000000,NaN,NaN,NaN
3,bias_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.05,6.938894e-18,4.814825e-35,1.387779e-16,0.050000,...,0.050000,0.000000,0.200000,-0.050000,0.000000,1.5,1.000000,1.500000,0.80,1.000000
4,bias_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.05,8.563758e-02,7.333795e-03,1.712752e+00,0.263158,...,0.000000,-0.736842,0.473684,0.000000,0.578947,NaN,0.000000,2.800000,NaN,6.500000



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/bias.py` or `composition.py`
- keep residue groups in `roxy/core/constants.py`
- expose a class such as `CompositionalBiasDescriptors`
- allow configurable:
  - residue-level inequality summaries
  - group-level skew summaries
  - selected asymmetry ratios
- add tests for:
  - empty sequences
  - nearly uniform sequences
  - single-residue dominated sequences
  - strongly biased charged sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_bias.to_csv("demo_compositional_bias_descriptors.csv", index=False)
